# Parquet and ORC: Columnar Storage, Compression, and Speedups

**Module**: 08_data_sources_and_formats  
**Notebook**: 02_parquet_and_orc.ipynb  
**Environment**: Apache Spark 4.2.1 · Standalone Cluster · PySpark · Python 3.13 · Java 21  

---

## Learning Objectives

- **What will I learn?**: Why Parquet and ORC are the gold standard for big data analytics: columnar storage, row-group statistics, dictionary encoding, and predicate pushdown.
- **Why does this topic exist?**: Switching from CSV to Parquet can reduce storage and scan time substantially, but the ratio depends on schema, compression, row-group layout, and query selectivity.
- **What problem does it solve?**: Inefficient full-table scans on uncompressed text formats.
- **Where is it used in real-world Data Engineering?**: Enterprise silver/gold lakehouse tables.

---

<!-- curriculum-enrichment:v1 -->

## Learning Contract

| Attribute | Value |
|---|---|
| **Priority** | Essential |
| **Estimated effort** | 2–3 h |
| **Prerequisites** | CSV/JSON, schemas, columnar storage basics |
| **Required depth** | Working knowledge: can choose a columnar format and explain pruning and compression |
| **Data Engineer** | Store analytical data in Parquet/ORC with sensible compression and partitioning. |
| **Senior Data Engineer** | Diagnose scan performance from file layout, row groups, and predicate pushdown. |
| **Spark specialist** | Reason about encodings, row-group sizing, and format-specific behavior. |
| **Next topic** | `08_data_sources_and_formats/03_partitioned_data_lake.ipynb` |

> **How to use this notebook**: read the problem first, then run the code, then connect the
> mechanism to the real-world cases and production notes. The goal is not to memorize API
> syntax; it is to be able to explain and defend a design decision.

---

## Problem and Motivation

### Why this concept exists

Parquet and ORC are columnar formats designed for analytical scans. They store values by column, compress
better than row-oriented text, and keep statistics that let readers skip row groups. The performance benefit is
not automatic: tiny files, poor row-group sizing, and missing predicates can erase most of the advantage.

### What happens if you ignore it

A team can still write code that runs on small data. The failure appears later: a job that
looks correct becomes slow, unstable, or expensive at scale, and the team lacks the vocabulary
to diagnose which Spark mechanism is responsible. This notebook builds that vocabulary and
ties it to observable behavior in the lab.

### A small mental model

Think of the topic as a contract between the logical work you describe and the physical work
Spark performs. The contract has a cost, a failure mode, and a set of trade-offs. The sections
below make each of those explicit.

---

## Core Concepts

### Definitions

| Term | Plain-language meaning | Technical meaning |
|---|---|---|
| **Columnar layout** | Store columns together | Read only referenced columns |
| **Row group** | Horizontal slice of a file | Unit of statistics and predicate pushdown |
| **Predicate pushdown** | Skip row groups | Uses min/max and null counts |
| **Column pruning** | Read only needed columns | Reduces I/O for wide tables |
| **Compression** | Reduce bytes on disk | Snappy, Zstandard, gzip, and others |
| **Encoding** | Represent values efficiently | Dictionary, run-length, bit-packing |

### How it works, step by step

1. The writer buffers rows and flushes row groups.
2. Each column chunk is encoded and compressed.
3. File and row-group statistics are written to the footer.
4. A reader prunes columns not referenced by the query.
5. It uses predicates and statistics to skip row groups.
6. It decompresses and decodes only the surviving data.

### What to keep in mind

- The mechanism is not free; it consumes CPU, memory, network, or disk.
- The right choice depends on data size, distribution, latency target, and failure tolerance.
- A plan or UI observation is stronger evidence than an intuition about what "should" happen.

## 2. Row vs Columnar Format
```
Row-Oriented (CSV):
[Row 0: 1001, 2026-01-03, North, Laptop] [Row 1: 1002, 2026-01-03, South, Phone]
(To read only'revenue', MUST scan all bytes across all columns!)

Columnar Format (Parquet):
[Column'order_id': 1001, 1002, ...]
[Column'region':   North, South, ...]  <-- Dictionary Encoded & Compressed!
(Reads ONLY the bytes of requested columns; skips unneeded row groups via min/max stats!)
```

### 1. Initialize SparkSession on Standalone Cluster

We connect to `spark://spark-master:7077` with the driver configured to bind to container hostname `jupyter`.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
import os
import time

spark = (
    SparkSession.builder
    .appName("Parquet_ORC_Lab")
    .master("spark://spark-master:7077")
    .config("spark.driver.host", "jupyter")
    .config("spark.driver.bindAddress", "0.0.0.0")
    .config("spark.eventLog.enabled", "true")
    .config("spark.eventLog.dir", "file:///opt/spark-events")
    .config("spark.sql.warehouse.dir", "file:///opt/spark-data/warehouse")
    .config("spark.hadoop.fs.permissions.umask-mode", "000")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

print(f" SparkSession initialized successfully!")
print(f"Spark Version:      {spark.version}")
print(f"Master URL:         {spark.sparkContext.master}")
print(f"App Name:           {spark.sparkContext.appName}")
print(f"Default Parallelism: {spark.sparkContext.defaultParallelism}")
print(f"Spark UI available at: http://localhost:4040")

In [ ]:
orders = spark.read.option("header", True).option("inferSchema", True).csv("/opt/spark-data/retail/orders.csv")
out_parquet = "/opt/spark-data/warehouse/orders_parquet"

# Write as Parquet
orders.write.mode("overwrite").parquet(out_parquet)
print("Written orders to Parquet successfully!")

# Read back and inspect
read_parquet = spark.read.parquet(out_parquet)
print(f"Parquet row count: {read_parquet.count()}")

---

## Experiment: Storage Size: CSV vs Parquet

- **1. What are we testing?**: Compare file size on disk between raw CSV and Snappy-compressed Parquet.
- **2. What do we expect?**: Parquet is 3x to 6x smaller than CSV.

In [ ]:
csv_size = os.path.getsize("/opt/spark-data/retail/orders.csv")
parquet_size = sum(os.path.getsize(os.path.join(out_parquet, f)) for f in os.listdir(out_parquet) if f.endswith(".parquet"))
print(f"Raw CSV Size:     {csv_size:,} bytes")
print(f"Parquet Size:     {parquet_size:,} bytes")
print(f"Compression Ratio: {csv_size / parquet_size:.2f}x smaller!")

**3. What happened?**: Parquet achieved massive compression savings.  
**4. Why did it happen?**: Columnar layout enables run-length encoding and dictionary compression.  
**5. Practical Engineering Lesson**: Never store historical data in CSV/JSON; always convert to Parquet.

---

## Real-World Use Cases

> The scenarios below are illustrative. Company names are omitted, and any numbers are hypothetical unless they are measured in this lab.

### Use Case 1: Analytics lake migration

- **Industry**: Retail
- **Scenario**: A team moves daily CSV extracts to Parquet for a BI workload.
- **Problem**: CSV scans read every column and row, and compression is poor.
- **Why this concept is relevant**: Parquet stores columns separately, so a query selecting five of fifty columns reads far less data.
- **Implementation idea**: Rewrite the extracts as partitioned Parquet and compare scanned bytes and runtime.
- **Expected result**: Faster queries and lower storage cost.
- **Alternative**: Keep CSV and use a query accelerator.
- **Trade-offs**: An accelerator helps but does not fix the underlying row-oriented layout.

### Use Case 2: Small-file accumulation

- **Industry**: IoT
- **Scenario**: Hourly ingestion creates thousands of tiny Parquet files.
- **Problem**: Each file has metadata and row-group overhead, and the reader spends time listing and opening files.
- **Why this concept is relevant**: Compaction into larger files restores the benefits of columnar storage and predicate pushdown.
- **Implementation idea**: Run a compaction job and compare file count, scan time, and task count.
- **Expected result**: Fewer files with better row-group utilization.
- **Alternative**: Use a table format with automatic compaction.
- **Trade-offs**: Automatic compaction reduces operations but adds a dependency and background work.

---

## Business Value

- Lower storage cost through compression.
- Faster BI queries from column pruning.
- Predictable scan performance with good file sizing.
- A format that most engines can read.

> **Note on numbers**: this repository does not claim production benchmarks. If you want to
> quantify a business case, measure the job in your own environment (runtime, shuffle bytes,
> spill, infrastructure cost) before and after the change.

---

## Production Engineering

### What changes when this runs in production

- Target reasonable file sizes; avoid thousands of tiny files.
- Use row-group sizes that match typical scans.
- Choose compression based on CPU/space trade-offs.
- Verify predicate pushdown in the plan.
- Do not expect columnar storage to fix a bad partitioning scheme.

### Investigation checklist

- [ ] Open the Spark UI and identify the relevant tab for this topic.
- [ ] Compare the metric before and after a configuration or code change.
- [ ] Check driver and executor logs for warnings or retries.
- [ ] Confirm that the fix is measured, not assumed.

### Common production mistakes

1. Copying a local-lab setting into a production cluster without sizing it.
2. Optimizing a single job without checking the downstream SLA and data-quality impact.
3. Treating a successful run as proof of correctness; verify row counts, keys, and invariants.
4. Forgetting that retries and backfills must be idempotent.

### Experiment: Measure column pruning

**Steps**

1. Write a wide Parquet table and query two columns.
2. Compare scanned bytes with a CSV version of the same data.
3. Inspect the scan's read schema and statistics.

**What to observe**: Parquet reads only the referenced columns, while CSV parses every row and column.

**Healthy result**: Scanned bytes are much lower for the columnar query.

**Problematic result**: The query still scans everything because it selects `*` or uses a non-pushdown predicate.

**Engineering takeaway**: Columnar storage rewards narrow projections.

---

## Trade-offs and Alternatives

### Advantages

- Column pruning
- Compression
- Predicate pushdown
- Wide engine support

### Limitations

- Not ideal for row-at-a-time updates
- Small files hurt
- Schema evolution is limited

### When to use it

- Analytical tables
- Data lakes
- BI extracts

### When not to use it

- Transactional updates
- Tiny files
- Frequent full-row rewrites

### Alternatives inside Spark

- Parquet
- ORC
- Vectorized readers

### Alternatives outside Spark

- Delta/Iceberg/Hudi
- Avro
- Warehouse-native formats

### Performance implications

Every choice has a cost. Measure runtime, shuffle read/write, spill, GC time, and task
duration percentiles before and after the change. A local four-core cluster is useful for
observing mechanics, but it is not a production benchmark.

---

## Practical Exercises

### Exercise 1 (Beginner): Read Parquet Subset
**Problem Statement**: Read only order_id and unit_price from Parquet.  
**Input / Context**: read_parquet

In [ ]:
# Write your solution for Exercise 1 here

### Exercise 2 (Intermediate): Verify Predicate Pushdown
**Problem Statement**: Filter Parquet on quantity > 10 and verify PushedFilters in plan.  
**Input / Context**: Provided dataset

In [ ]:
# Write your solution for Exercise 2 here

### Exercise 3 (Advanced): Snappy vs ZSTD Compression
**Problem Statement**: Write with compression codec'zstd'.  
**Input / Context**: Provided dataset

In [ ]:
# Write your solution for Exercise 3 here

---

## Production Debugging Exercise

### Parquet slower than CSV

**Scenario**: A Parquet table is slower to query than the original CSV.

**Your task**: identify the most likely root cause, propose one diagnostic step, and propose
one fix. Write your answer before opening the model approach.

**Hints**

1. Check the number and size of files.
2. Look for a full scan caused by `select('*')` or a cast.
3. Inspect row-group size and compression codec.

<details>
<summary>Model approach (open after trying)</summary>

1. Reproduce the issue with the smallest possible dataset.
2. Use the Spark UI and the physical plan to locate the stage or operator that dominates.
3. Change one variable at a time and re-measure.
4. Record the before/after metric so the fix is defensible.

The exact fix depends on the evidence. The important habit is to separate allocation,
scheduling, shuffle, memory, and data-quality causes instead of guessing.

</details>

---

## Reference Solutions

<details>
<summary><b>Click to expand full solutions</b></summary>

#### Solution 1 (Read Parquet Subset):
```python
spark.read.parquet(out_parquet).select('order_id','unit_price').show(3)
```

#### Solution 2 (Verify Predicate Pushdown):
```python
spark.read.parquet(out_parquet).filter('quantity > 10').explain()
```

#### Solution 3 (Snappy vs ZSTD Compression):
```python
orders.write.mode('overwrite').option('compression','zstd').parquet('/opt/spark-data/warehouse/orders_zstd')
```

</details>

---

## Module Summary & Interview Guide

### Key Takeaways
- **Columnar Format**: Compresses identical types together; supports column pruning and predicate pushdown.
- **Standard**: Parquet is Spark's native format.

### Essential Technical Terms
- `Row Group`: Chunk of columnar data with min/max statistics.
- `Snappy`: Default fast compression codec.

### Common Engineering Mistakes
- **Using CSV in Production**: Wastes storage and kills query performance.

### Interview Questions on This Topic
- **Q: How does Parquet skip reading data?**
  - *A*: Through row group metadata storing min/max statistics for each column, allowing Spark to skip entire blocks if the filter range does not intersect.


### More Interview Questions

- **Q: Why is Parquet faster for analytical queries?**
  - *A*: It stores data by column, so a query reads only referenced columns, and it uses compression plus row-group statistics to reduce I/O.
- **Q: What is predicate pushdown in Parquet?**
  - *A*: Using row-group statistics and filters to skip row groups that cannot contain matching rows, reducing the data read.

### What to Study Next
Open 03_partitioned_data_lake.ipynb

---

### Tear Down SparkSession
Always close your session when finishing to release executor threads and driver ports.

In [ ]:
spark.stop()
print('SparkSession successfully stopped.')